# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tkg-create/FlyRank-ML-Track/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [18]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/tkg-create/FlyRank-ML-Track"
REPO_DIR = "FlyRank-ML-Track"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/FlyRank-ML-Track/FlyRank-ML-Track/FlyRank-ML-Track
Starter data found. You're ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane (Refresh/Content Opportunity Scoring) is a ranking/scoring task, not classification or clustering. The output a reviewer needs is an ordered list of pages by priority, not a hard yes/no bucket — a page just above the cutoff and one just below it aren't meaningfully different categories, just slightly different priorities. Clustering doesn't fit either, since the goal is a per-page priority order, not unlabeled groups. Scoring/ranking is the natural fit for "which 50 should I look at first."

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

What I'm predicting is urgency — which pages most need review, not just yes/no. The target is is_declining_label (trend_direction == "down"), the closest available proxy since there is no urgency label. The model outputs a decline probability per page, which combines with other signals into a score, an action, and reason codes — together forming the actual deliverable: a ranked top-50 queue of what pages most urgently need review and why. Since trend_direction is computed from trend_pct, itself built from impressions_last_30d vs. impressions_prev_30d, all four columns are excluded as features — including them would let the model reconstruct the label instead of learning a pattern.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My success metric is precision@50 — of the top 50 pages the model surfaces, what fraction are actually declining? This maps directly onto the real constraint a reviewer works under: they only have capacity for about 50 pages a week, so the number that matters is "how often can I trust this exact list," not overall ranking quality across every possible cutoff.

A good number here is one that clearly beats the hand-rule baseline as that gap is what justifies using a model at all, rather than the existing rule. Recall isn't a useful metric in this setting: with thousands of declining pages in the dataset, a fixed top-50 list can never recall more than a tiny fraction of them, so that number would be low regardless of model quality and wouldn't reflect anything meaningful about performance.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Each row is a single page, with the columns mostly being information about the page. Tthe best way to show this is via pointing out a few columns, those being content_type, word_count, and ctr — values that describe one distinct piece of content, not a summary across a client or a time period.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Unit of analysis: one row = one page (content item)
print(f"Shape: {df.shape}")
print(f"One row = one page. {df['content_id'].nunique()} unique content_ids across {df['client_id'].nunique()} clients.")

df.head()

Shape: (30000, 44)
One row = one page. 30000 unique content_ids across 32 clients.


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [23]:
# Noted Columns
df[["content_id", "content_type", "word_count", "ctr"]].head()

,content_id,content_type,word_count,ctr
0,content_304f48230142,keyword article,3221.0,0.76
1,content_a1fb4e703a9e,keyword article,2481.0,0.05
2,content_9aa793d4d895,keyword article,3515.0,0.09
3,content_331d6c4de07b,keyword article,NaN,0.49
4,content_d99b7a2d90ca,keyword article,2803.0,0.13


In [24]:
# Build the target
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Target: is_declining_label")
print(df["is_declining_label"].value_counts(normalize=True).round(3))

Target: is_declining_label
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule struggles here because urgency isn't explained by any single signal, it comes from how several combine. The freshness gap between declining and stable pages has previously been observed to be small (51.6 vs. 49.1 days), so something like staleness alone doesn't separate them, but the position-tier breakdown showed declining pages spread across tiers rather than clustered in one.

This indicates no single threshold (e.g. "is it stale" or "is it low-ranked") reliably captures who's actually at risk, which is consistent with the feature importances observed earlier, where no one feature dominates — days_with_impressions, log_impressions_90d, and avg_position all carry meaningful weight, with several more contributing smaller amounts.

A hand rule can only encode a handful of thresholds at most before it becomes unmanageable while a model can weigh a dozen signals against each other simultaneously, which is exactly the gap the baseline-vs-model comparison in an earlier Notebook demonstrated — though that comparison predates the lane split, so it's evidence the general approach works rather than a number confirmed for this lane specifically.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

---



Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.